In [1]:
!pip install datasets tqdm pandas
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# Kaggle doesn't allow & so use subprocess instead
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

!ollama pull aya:8b




The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 133 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (2,111 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
Selecting previously unselected package zstd.
(Reading database ... 124626 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
###

In [2]:
from datasets import load_dataset
from tqdm.notebook import tqdm
import pandas as pd
import requests

In [3]:
path = "/kaggle/input/datasets/feryaljadallah/fullbayyin/full_bayyin_dataset.csv"
OLLAMA_URL = "http://localhost:11434/api/generate"

In [4]:
import pandas as pd
from datasets import Dataset

df_raw = pd.read_csv(path, dtype=str)  # force everything to string, no inference
dataset = Dataset.from_pandas(df_raw)
print(f"Loaded {len(dataset)} rows")
print(f"Columns: {dataset.column_names}")

Loaded 71044 rows
Columns: ['ID', 'Sentence', 'Word_Count', 'Word', 'Lex', 'D3Tok', 'D3Lex', 'Readability_Level', 'Document', 'Source', 'Book', 'Author', 'Domain', 'Text_Class', 'Dataset_Source']


In [5]:
from collections import Counter
level_counts = Counter(int(row.get("Readability_Level", 0) or 0) for row in dataset)
for lvl in sorted(level_counts):
    print(f"Level {lvl}: {level_counts[lvl]} sentences")

Level 1: 9027 sentences
Level 2: 7735 sentences
Level 3: 14369 sentences
Level 4: 17973 sentences
Level 5: 13109 sentences
Level 6: 8831 sentences


In [6]:
sentences = dataset["Sentence"]

In [7]:
def simplify_with_llm(sentence, target_level):
    level_instructions = {
        1: "بشكل بسيط جداً مناسب للأطفال، باستخدام كلمات قصيرة وشائعة فقط",
        2: "بشكل بسيط وواضح، مناسب للقراء المبتدئين",
        3: "بشكل مبسط قليلاً، مع الحفاظ على الأسلوب العربي الفصيح"
    }

    prompt = f"""أنت متخصص في تبسيط النصوص العربية. مهمتك تبسيط الجملة التالية {level_instructions[target_level]}.

قواعد مهمة:
- أعد كتابة الجملة فقط، بدون أي مقدمة أو شرح
- حافظ على المعنى الأصلي
- لا تضف معلومات جديدة
- اكتب جملة واحدة فقط

الجملة الأصلية: {sentence}

الجملة المبسّطة:"""

    response = requests.post(OLLAMA_URL, json={
        "model": "aya:8b",
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.3, "num_predict": 256}
    })
    return response.json()["response"].strip()

print("Ollama client ready.")

Ollama client ready.


In [8]:
def is_hard_level(level):
    return level in [5, 6]

In [9]:
def simplify_sentence_levels(sentence):
    results = {}
    for lvl in [1, 2, 3]:
        try:
            simplified = simplify_with_llm(sentence, lvl)
            results[lvl] = simplified
        except Exception as e:
            print(f"  Level {lvl} failed: {e}")
    return results if results else None

In [10]:
def is_valid(original, simplified):
    if not simplified or len(simplified.strip()) < 3:
        return False

    # Strip any residual prefixes
    prefixes_to_clean = ["الجملة المبسّطة:", "إليك", "أعد كتابة", "الترجمة:", "ببساطة:"]
    for p in prefixes_to_clean:
        simplified = simplified.replace(p, "").strip()

    if len(simplified) > len(original) * 2.5:
        return False

    if simplified.strip() == original.strip():
        return False

    return True

In [11]:
def save_checkpoint(data, batch_num):
    pd.DataFrame(data).to_csv(f"simplified_checkpoint_{batch_num}.csv", index=False)
    print(f"  Checkpoint saved: {len(data)} rows")

In [12]:
PART = "A"

# Remove Quran sentences, then split
hard_rows = [
    (i, row) for i, row in enumerate(dataset)
    if is_hard_level(int(row.get("Readability_Level", 0) or 0))
    and row.get("Source") != "Quran"
]

print(f"Total hard non-Quran sentences: {len(hard_rows)}")

mid = len(hard_rows) // 2
hard_rows = hard_rows[:mid] if PART == "A" else hard_rows[mid:]

# Resume from last checkpoint
import os
checkpoint_file = "/kaggle/input/datasets/feryaljadallah/bayyinpartacheckpoint/bayyin_simplified_partA_checkpoint_10000.csv"

if os.path.exists(checkpoint_file):
    simplified_data = pd.read_csv(checkpoint_file, dtype=str).to_dict("records")
    already_done = len(simplified_data) // 3
    hard_rows = hard_rows[already_done:]
    print(f"Resuming from {already_done} sentences, {len(hard_rows)} remaining")
else:
    simplified_data = []
    print("No checkpoint found, starting fresh")

simplified_data_grouped = []
SAVE_EVERY = 100

for count, (orig_idx, row) in enumerate(tqdm(hard_rows)):
    sentence = row["Sentence"]

    simplified_versions = simplify_sentence_levels(sentence)
    if not simplified_versions:
        continue

    entry = {"Original_Sentence": sentence, "Simplified": {}}

    for lvl, simp in simplified_versions.items():
        clean = simp.strip()
        if is_valid(sentence, clean):
            entry["Simplified"][lvl] = clean
            new_row = dict(row)
            new_row["Original_Sentence"] = sentence
            new_row["Sentence"] = clean
            new_row["Readability_Level"] = str(lvl)
            simplified_data.append(new_row)

    if entry["Simplified"]:
        simplified_data_grouped.append(entry)

    if (count + 1) % SAVE_EVERY == 0:
        pd.DataFrame(simplified_data).to_csv(
            f"bayyin_simplified_part{PART}_checkpoint_latest.csv", index=False
        )
        print(f"  Checkpoint saved: {len(simplified_data)} rows")

print(f"Done. Generated {len(simplified_data)} rows from {len(simplified_data_grouped)} sentences.")

Total hard non-Quran sentences: 21739
Resuming from 9962 sentences, 907 remaining


  0%|          | 0/907 [00:00<?, ?it/s]

  Checkpoint saved: 30187 rows
  Checkpoint saved: 30487 rows
  Checkpoint saved: 30785 rows
  Checkpoint saved: 31085 rows
  Checkpoint saved: 31384 rows
  Checkpoint saved: 31682 rows
  Checkpoint saved: 31981 rows
  Checkpoint saved: 32278 rows
  Checkpoint saved: 32573 rows
Done. Generated 32594 rows from 907 sentences.


In [13]:
df = pd.DataFrame(simplified_data)
df.to_csv(f"bayyin_simplified_part{PART}_final.csv", index=False)
print(f"Saved {len(df)} rows to bayyin_simplified_part{PART}_final.csv")
print(df.head())

Saved 32594 rows to bayyin_simplified_partA_final.csv
            ID                                           Sentence Word_Count  \
0  10100300018  "عدم الانحياز" يعني أن تكون محايداً ولا تفضل أ...         12   
1  10100300018  "عدم الانحياز" هو مصطلح سياسي يعني أن تكون محا...         12   
2  10100300018     "عدم الانحياز" هو مصطلح سياسي يشير إلى الحياد.         12   
3  10100300019  الدول غير المنحازة لا تتعامل مع دول الشرق أو ا...         15   
4  10100300019  تلتزم الدول غير المنحازة بعدم التحالف مع دول ا...         15   

                                                Word  \
0  • • " عدم الانحياز " اصطلاح سياسي يعنى الحياد . .   
1  • • " عدم الانحياز " اصطلاح سياسي يعنى الحياد . .   
2  • • " عدم الانحياز " اصطلاح سياسي يعنى الحياد . .   
3  حيث تلتزم الدول غير المنحازة بعدم الارتباط بأي...   
4  حيث تلتزم الدول غير المنحازة بعدم الارتباط بأي...   

                                                 Lex  \
0       • • " عدم ٱنحياز " ٱصطلاح سياسي عنى حياد . .   
1       • • " عد

In [14]:
for entry in simplified_data_grouped[:5]:
    print("Original:  ", entry["Original_Sentence"])
    for lvl in sorted(entry["Simplified"]):
        label = {1: "Level 1 (simplest)", 2: "Level 2", 3: "Level 3"}[lvl]
        print(f"  {label}: {entry['Simplified'][lvl]}")
    print("-" * 60)

Original:   الفرع الاول. الحقوق المدنية والسياسية
  Level 1 (simplest): الحقوق المدنية والسياسية
  Level 2: الحقوق المدنية والسياسية
  Level 3: الحقوق المدنية والسياسية.
------------------------------------------------------------
Original:   العراقيون متساوون أمام القانون دون تمييز بسبب الجنس أو العرق أو القومية أو الاصل أو اللون أو الدين أو المذهب أو المعتقد أو الرأي أو الوضع الاقتصادي أو الاجتماعي .
  Level 1 (simplest): الجميع في العراق سواسية أمام القانون، ولا أحد يُعامل بشكل مختلف بسبب جنسه أو لونه أو دينه.
  Level 2: العراقيون متساوون أمام القانون دون أي تمييز.
  Level 3: العراقيون متساوون أمام القانون دون تمييز بينهم على أساس الجنس أو العرق أو القومية أو الأصل أو اللون أو الدين أو المذهب أو المعتقد أو الرأي أو الوضع الاقتصادي أو الاجتماعي.
------------------------------------------------------------
Original:   لكل فرد الحق في الحياة والأمن والحرية، ولايجوز الحرمان من هذه الحقوق أو تقييدها إلا وفقاً للقانون، وبناءً على قرار صادر من جهة قضائية مختصة .
  Level 1 (simplest): لكل ش